# Feature Engineering

This notebook demonstrates the feature engineering process:
- SOC-based features
- Efficiency metrics
- Cost features
- Time-based features
- User-level aggregated features
- Interaction features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.utils import load_data
from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Load and preprocess data
df = load_data("ev_charging_data.csv")
preprocessor = DataPreprocessor()
df_processed = preprocessor.preprocess(df)

print(f"Original shape: {df.shape}")
print(f"Processed shape: {df_processed.shape}")

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer()

# Engineer features
df_features = feature_engineer.engineer_features(df_processed.copy(), encode_categorical=True)

print(f"Shape after feature engineering: {df_features.shape}")
print(f"\nNew features created:")
print([col for col in df_features.columns if col not in df_processed.columns][:20])

In [ ]:
# Examine SOC features
soc_features = [col for col in df_features.columns if 'soc' in col.lower()]
print("SOC-related features:")
for feat in soc_features:
    print(f"  - {feat}")
    if feat in df_features.columns:
        print(f"    Mean: {df_features[feat].mean():.4f}, Std: {df_features[feat].std():.4f}")

In [ ]:
# Examine efficiency features
efficiency_features = [col for col in df_features.columns if 'efficiency' in col.lower() or 'utilization' in col.lower()]
print("Efficiency-related features:")
for feat in efficiency_features:
    print(f"  - {feat}")
    if feat in df_features.columns:
        print(f"    Mean: {df_features[feat].mean():.4f}")

In [ ]:
# Examine cost features
cost_features = [col for col in df_features.columns if 'cost' in col.lower() and col != 'Charging Cost (USD)']
print("Cost-related features:")
for feat in cost_features:
    print(f"  - {feat}")
    if feat in df_features.columns:
        print(f"    Mean: {df_features[feat].mean():.4f}")

In [ ]:
# Examine time features
time_features = [col for col in df_features.columns if col in ['hour', 'day_of_week', 'month', 'is_weekend', 'is_peak_hours', 'is_off_peak', 'season']]
print("Time-related features:")
for feat in time_features:
    if feat in df_features.columns:
        print(f"  - {feat}")
        if df_features[feat].dtype in ['int64', 'float64']:
            print(f"    Mean: {df_features[feat].mean():.4f}")
        else:
            print(f"    Unique values: {df_features[feat].nunique()}")

In [ ]:
# Examine user features
user_features = [col for col in df_features.columns if 'user' in col.lower()]
print("User-level aggregated features:")
for feat in user_features:
    print(f"  - {feat}")
    if feat in df_features.columns:
        print(f"    Mean: {df_features[feat].mean():.4f}, Std: {df_features[feat].std():.4f}")

In [ ]:
# Feature importance visualization (correlation with target)
target = 'Charging Cost (USD)'
feature_cols = feature_engineer.get_feature_columns(df_features, target_col=target)

correlations = df_features[feature_cols + [target]].corr()[target].sort_values(ascending=False)
correlations = correlations[correlations.index != target]

plt.figure(figsize=(10, 12))
top_corr = correlations.head(20)
sns.barplot(y=top_corr.index, x=top_corr.values, palette='viridis')
plt.xlabel('Correlation with Charging Cost')
plt.title('Top 20 Features Correlated with Charging Cost')
plt.tight_layout()
plt.show()